# Notebook 52 — First-Principles Derivation from `S(x) = exp(-∫ Γ(u) du)`

This notebook follows Notebook 51.

Notebook 51 built an analytic approximation for the stretched-exponential
exponent `b` from geometric features of the effective-rate process `Γ_eff(x)`.

Here we take the next step:

> start from the exact relation
>
> `S(x) = exp(-∫₀ˣ Γ(u) du)`
>
> and derive why non-constant `Γ(u)` naturally generates stretched-exponential behavior.

## Main goals

1. start from the integral expression for `S(x)`,
2. test several analytic families of `Γ(x)`,
3. compare the exact integral to stretched-exponential fits,
4. derive approximate relations between `Γ(x)` shape and the exponent `b`,
5. connect the derivation to the earlier notebooks.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


## Core definitions

In [ ]:
x = np.linspace(1e-4, 1.0, 600)
dx = x[1] - x[0]

def stretched(x, a, b):
    return np.exp(-a * np.power(x, b))

def fit_stretched(x, S):
    popt, _ = curve_fit(
        stretched,
        np.clip(x, 1e-12, None),
        np.clip(S, 1e-12, 1.0),
        p0=[1.0, 1.0],
        bounds=([0.0, 0.1], [100.0, 3.0]),
        maxfev=50000,
    )
    a, b = [float(v) for v in popt]
    pred = stretched(x, a, b)
    mse = float(np.mean((S - pred) ** 2))
    return a, b, pred, mse

def build_S_from_gamma(gamma_x, x):
    integral = np.cumsum(gamma_x) * (x[1] - x[0])
    return np.exp(-integral)


## Exact derivation for a power-law rate

Take

`Γ(x) = c x^(m-1)`

Then

`∫₀ˣ Γ(u) du = (c/m) x^m`

so

`S(x) = exp(-(c/m) x^m)`

which is *exactly* a stretched exponential with

- `a = c/m`
- `b = m`

This is the cleanest first-principles bridge from rate process to universality exponent.


In [ ]:
def gamma_power_exact(x, c=1.8, m=0.85):
    return c * np.power(x, m - 1.0)

power_params = [
    (1.5, 0.75),
    (1.8, 0.90),
    (2.0, 1.10),
]

rows_exact = []
for c, m in power_params:
    g = gamma_power_exact(x, c=c, m=m)
    S = build_S_from_gamma(g, x)
    a_fit, b_fit, S_fit, mse = fit_stretched(x, S)
    rows_exact.append({
        "c": c, "m_true": m,
        "gamma": g, "S": S,
        "a_fit": a_fit, "b_fit": b_fit,
        "S_fit": S_fit, "mse": mse,
    })

for row in rows_exact:
    print(
        f'c={row["c"]:.2f}, m_true={row["m_true"]:.3f}, '
        f'b_fit={row["b_fit"]:.3f}, mse={row["mse"]:.3e}'
    )


## Plot 1 — Exact power-law rates

In [ ]:
plt.figure(figsize=(8.0, 5.0))
for row in rows_exact:
    plt.plot(x, row["gamma"], label=f'm={row["m_true"]:.2f}')
plt.xlabel("x")
plt.ylabel("Γ(x)")
plt.title("Power-law rate families Γ(x) = c x^(m-1)")
plt.legend()
plt.tight_layout()
plt.show()


## Plot 2 — Exact stretched-exponential decays from the integral law

In [ ]:
plt.figure(figsize=(8.0, 5.0))
for row in rows_exact:
    plt.plot(x, row["S"], label=f'exact S, m={row["m_true"]:.2f}')
    plt.plot(x, row["S_fit"], linestyle="--", label=f'fit b={row["b_fit"]:.2f}')
plt.xlabel("x")
plt.ylabel("S(x)")
plt.title("Exact stretched exponentials generated by power-law Γ(x)")
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


## Local exponent identity

For a stretched exponential

`S(x) = exp(-a x^b)`

we have

`-log S(x) = a x^b`

Taking a logarithm again:

`log(-log S(x)) = log a + b log x`

So the local slope in `(log x, log(-log S))` space is the effective exponent `b`.

This gives a first-principles way to *measure* whether a decay law is locally stretched-exponential.


In [ ]:
def local_b_from_S(x, S):
    eps = 1e-12
    y = np.log(np.clip(-np.log(np.clip(S, eps, 1.0)), eps, None))
    z = np.log(np.clip(x, eps, None))
    return np.gradient(y, z)

plt.figure(figsize=(8.2, 5.0))
for row in rows_exact:
    b_local = local_b_from_S(x, row["S"])
    plt.plot(x, b_local, label=f'm={row["m_true"]:.2f}')
plt.axhline(1.0, linestyle="--", linewidth=1)
plt.xlabel("x")
plt.ylabel("local exponent")
plt.title("Local exponent extracted from exact S(x)")
plt.legend()
plt.tight_layout()
plt.show()


## Beyond exact power laws: slowly varying corrections

Now take

`Γ(x) = c x^(m-1) [1 + ε h(x)]`

Then

`∫₀ˣ Γ(u) du = (c/m) x^m + correction`

so the decay is no longer *exactly* stretched exponential, but it remains close,
with an effective exponent controlled by the dominant power `m` and corrected by the shape term `h(x)`.

This is the natural analytic explanation for why:
- Notebook 44 found a stretched law,
- Notebook 48 found structured corrections,
- Notebook 49 found functional dependence,
- Notebook 50 learned the map,
- Notebook 51 fit an interpretable approximation.


In [ ]:
def gamma_corrected(x, c=1.8, m=0.9, eps=0.25, kind="linear"):
    base = c * np.power(x, m - 1.0)
    if kind == "linear":
        h = x
    elif kind == "quadratic":
        h = (x - 0.5)**2
    elif kind == "wave":
        h = np.sin(2*np.pi*x)
    else:
        h = np.zeros_like(x)
    return base * (1.0 + eps * h)

corr_specs = [
    ("linear", 0.20),
    ("quadratic", 0.30),
    ("wave", 0.15),
]

rows_corr = []
for kind, eps in corr_specs:
    g = gamma_corrected(x, c=1.8, m=0.9, eps=eps, kind=kind)
    S = build_S_from_gamma(g, x)
    a_fit, b_fit, S_fit, mse = fit_stretched(x, S)
    rows_corr.append({
        "kind": kind, "eps": eps,
        "gamma": g, "S": S,
        "a_fit": a_fit, "b_fit": b_fit,
        "S_fit": S_fit, "mse": mse,
    })

for row in rows_corr:
    print(
        f'{row["kind"]}: eps={row["eps"]:.2f}, '
        f'b_fit={row["b_fit"]:.3f}, mse={row["mse"]:.3e}'
    )


## Plot 3 — Corrected Γ(x) families

In [ ]:
plt.figure(figsize=(8.0, 5.0))
for row in rows_corr:
    plt.plot(x, row["gamma"], label=f'{row["kind"]}, eps={row["eps"]:.2f}')
plt.xlabel("x")
plt.ylabel("Γ(x)")
plt.title("Power-law rates with structured corrections")
plt.legend()
plt.tight_layout()
plt.show()


## Plot 4 — Corrected decays and stretched fits

In [ ]:
plt.figure(figsize=(8.2, 5.2))
for row in rows_corr:
    plt.plot(x, row["S"], label=f'{row["kind"]} data')
    plt.plot(x, row["S_fit"], linestyle="--", label=f'{row["kind"]} fit (b={row["b_fit"]:.2f})')
plt.xlabel("x")
plt.ylabel("S(x)")
plt.title("Structured corrections preserve approximate stretched universality")
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


## First-order geometric approximation

For the corrected families, the fitted exponent is approximately controlled by:

- dominant power `m`
- average slope of `Γ(x)`
- average curvature of `Γ(x)`
- coefficient of variation (CV)

This motivates the analytic ansatz:

`b ≈ α + β⟨|dΓ/dx|⟩ + γ⟨|d²Γ/dx²|⟩ + δ CV`

which is exactly what Notebook 51 tested numerically.


In [ ]:
def gamma_features(gamma_x, x):
    d1 = np.gradient(gamma_x, x)
    d2 = np.gradient(d1, x)
    mean = float(np.mean(gamma_x))
    std = float(np.std(gamma_x))
    cv = float(std / mean)
    return {
        "slope_abs": float(np.mean(np.abs(d1))),
        "curvature_abs": float(np.mean(np.abs(d2))),
        "cv": cv,
    }

feat_rows = []
for row in rows_corr:
    feats = gamma_features(row["gamma"], x)
    feat_rows.append({
        "kind": row["kind"],
        "b_fit": row["b_fit"],
        **feats
    })

for row in feat_rows:
    print(row)


In [ ]:
X = np.array([[r["slope_abs"], r["curvature_abs"], r["cv"]] for r in feat_rows], dtype=float)
y = np.array([r["b_fit"] for r in feat_rows], dtype=float)

X_aug = np.column_stack([np.ones(len(X)), X])
coeffs = np.linalg.lstsq(X_aug, y, rcond=None)[0]
y_pred = X_aug @ coeffs

print("Analytic coefficients:")
print("alpha, beta, gamma, delta =", coeffs)

plt.figure(figsize=(6.8, 4.8))
plt.scatter(y, y_pred, s=80)
lims = [min(np.min(y), np.min(y_pred)) - 0.01, max(np.max(y), np.max(y_pred)) + 0.01]
plt.plot(lims, lims, linestyle="--")
for i, row in enumerate(feat_rows):
    plt.annotate(row["kind"], (y[i], y_pred[i]), fontsize=8, xytext=(4,4), textcoords="offset points")
plt.xlabel("true b")
plt.ylabel("analytic approximation")
plt.title("First-order geometric approximation for b")
plt.tight_layout()
plt.show()


## Summary of the derivation

### Exact result
If

`Γ(x) = c x^(m-1)`

then

`S(x) = exp(-(c/m) x^m)`

so the exponent is exactly

`b = m`.

### Corrected result
If

`Γ(x) = c x^(m-1) [1 + ε h(x)]`

then the same power `m` still controls the dominant stretched behavior,
while the correction `h(x)` perturbs the fitted exponent.

### Consequence
This gives a first-principles explanation for the empirical law:

- dominant rate scaling sets the main exponent,
- geometric features of `Γ(x)` shift the observed `b`.

That is why a feature-based approximation in Notebook 51 and a learned map in Notebook 50 both work.


## Final conclusion

This notebook provides the first-principles derivation layer of the project.

It shows that:

1. stretched exponentials arise exactly from power-law effective-rate processes,
2. structured deviations from pure power laws generate controlled shifts in `b`,
3. the exponent can therefore be approximated from geometric features of `Γ_eff(x)`.

So the chain is now complete:

`Γ_eff(x)`  
→ `S(x) = exp(-∫ Γ(u) du)`  
→ stretched-exponential law  
→ structured correction to `b`  
→ analytic / learned universality map
